<a href="https://colab.research.google.com/github/BlazeCharm/ITA-Project/blob/main/ITA_Main_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from google.colab import files
import io

# --- STEP 1: UPLOAD & LOAD ---
print("Upload your 'newsCorpora.csv' dataset:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

cols = ['ID', 'TITLE', 'URL', 'PUBLISHER', 'CATEGORY', 'STORY', 'HOSTNAME', 'TIMESTAMP']
news_df = pd.read_csv(io.BytesIO(uploaded[filename]), sep='\t', names=cols)

# Using all available business headlines (~115,000 rows)
business_news = news_df[news_df['CATEGORY'] == 'b'].copy()
business_news['DATE'] = pd.to_datetime(business_news['TIMESTAMP'], unit='ms').dt.date

# --- STEP 2: STAGE 1 - UNSUPERVISED ---
print(f"Running Topic Modeling on {len(business_news)} headlines...")
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X_text = vectorizer.fit_transform(business_news['TITLE'])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
business_news['TOPIC_CLUSTER'] = kmeans.fit_predict(X_text)

daily_sentiment = business_news.groupby('DATE')['TOPIC_CLUSTER'].agg(lambda x: x.value_counts().index[0]).reset_index()

# --- STEP 3: FETCH & REINDEX FINANCIAL DATA ---
start_date = daily_sentiment['DATE'].min()
end_date = daily_sentiment['DATE'].max()
print(f"Total calendar days found in news: {len(daily_sentiment)}")

# THE FIX: 300-day buffer ensures SMA_200 has enough history to calculate
print("Fetching and aligning S&P 500 data...")
spy = yf.download('SPY', start=start_date - pd.Timedelta(days=300), end=end_date + pd.Timedelta(days=2))

# Flatten columns if MultiIndex (yfinance update fix)
if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)

# Reindex to include all calendar days (Forward-fill the weekends)
all_days = pd.date_range(start=spy.index.min(), end=spy.index.max(), freq='D')
spy = spy.reindex(all_days).ffill()

# Feature Engineering
spy['SMA_50'] = spy['Close'].rolling(window=50).mean()
spy['SMA_200'] = spy['Close'].rolling(window=200).mean()
spy['Price_Change'] = spy['Close'].pct_change()

# Target: 1 if volatility > 0.5%, else 0
spy['VOLATILITY_TARGET'] = (spy['Price_Change'].abs() > 0.005).astype(int)

# Drop NaNs now that the moving averages have enough runway
spy = spy.dropna()

# --- STEP 4: FEATURE JOIN ---
spy.index = pd.to_datetime(spy.index).date
final_df = pd.merge(spy, daily_sentiment, left_index=True, right_on='DATE')

# --- STEP 5: STAGE 2 - SUPERVISED ---
if final_df.empty:
    print("Error: No overlapping dates found. Check dataset alignment.")
else:
    print(f"Successfully merged! Training on {len(final_df)} overlapping days.")
    X = final_df[['SMA_50', 'SMA_200', 'TOPIC_CLUSTER']]
    y = final_df['VOLATILITY_TARGET']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)

    print("\n" + "="*40)
    print("FINAL HYBRID MODEL RESULTS")
    print("="*40)
    print(classification_report(y_test, rf.predict(X_test)))

Upload your 'newsCorpora.csv' dataset:
